# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "C:\\Users\\Administrator\\Documents\\coding\\comp5329\\ass1"
# PROJECT_ROOT = '/opt/jupyter/code/comp5329-ass01-main'

In [2]:
# Install Python dependencies (run once per session)
# !pip install -r {PROJECT_ROOT}/requirements.txt -q
# !python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [3]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: C:\Users\Administrator\Documents\coding\comp5329\ass1


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [4]:
# from Tools.download import download_mini,download_squad,download_glove

# # download_mini(data_dir="_data")
# download_squad(squad_dir="_data/squad")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [5]:
# from Tools.preproc import preprocess
# print('-----------')
# preprocess(
#     train_file="_data/squad/train-mini.json",
#     dev_file="_data/squad/dev-v1.1.json",
#     glove_word_file="_data/glove/glove.mini.txt",
#     target_dir="_data",
#     para_limit=400,
#     ques_limit=50,
# )
# print('-----------')

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [7]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    # num_steps  = 60000,
    num_steps = 10000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "adam",#"sgd",
    scheduler_name = 'step',#"none",
    loss_name      = "qa_nll",
    
    checkpoint=200,
    momentum = 0.9,
    # dropout=0.2,
    # dropout_char=0.1,
    # init_name = 'xavier_normal', #'kaiming_normal'
    
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:41<00:00,  4.86it/s]


STEP      200  loss 9796275511083666.000000



100%|██████████| 150/150 [00:08<00:00, 18.28it/s]


VALID(train) loss 998399883055814.000000  F1 6.203999  EM 0.000000



100%|██████████| 150/150 [00:08<00:00, 18.23it/s]


TEST        loss 1016917374393821.875000  F1 5.929489  EM 0.250000

Learning rate: [1.0]


 72%|███████▏  | 143/200 [00:30<00:12,  4.67it/s]


KeyboardInterrupt: 


print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 1309/1309 [00:22<00:00, 58.01it/s]


TEST  loss 2.756404  F1 35.112668  EM 24.634496
F1: 35.1127  |  EM: 24.6345  |  Loss: 2.756404
